In [1]:
from syto.data.atlases.celfieish_atlases import CpGBetaCountsMethylationAtlas
import pandas as pd
import os
import json
import numpy as np

In [1]:
data_path = "~/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041"

In [2]:
data_path = "~/Data/Loyfer/SimulatedReads_205files_hg38/all_reads_transformed_uxm_prepared.parquet"

In [ ]:
# data = [pd.read_parquet(os.path.join(data_path,f"{x}.parquet")) for x in ["train", "valid", "test"] ]

In [ ]:
# data = pd.concat(data)

In [3]:
data = pd.read_parquet(data_path)

In [4]:
labels_dict_path = "../../../App/labels_dict.json"
with open(labels_dict_path, "r", encoding="utf-8") as f:
            # JSON keys are strings; convert to {int: str}
            labels_dict = json.load(f)

In [5]:
data.reset_index(inplace=True)

In [6]:
data.rename(columns = {"chr":"chromosome", "trimmed_start":"read_start", "trimmed_end":"read_end"}, inplace=True)

In [7]:
atlas = CpGBetaCountsMethylationAtlas.from_reads(data, reference_genome="hg38", atlas_name="UXMU25_hg38_l1", labels_dict=labels_dict, output_path="/home/luna.kuleuven.be/u0169940/Repos/syto/baselines/deconvolution/celfieish/UXMU25_hg38_l1.txt")

Aggregating groups: 100%|██████████| 36457/36457 [01:16<00:00, 478.21group/s]


In [15]:
from baselines.deconvolution.celfieish import build_celfieish_input, celfieish_deconvolution
from baselines.deconvolution.celfie import build_celfie_input, celfie_deconvolution

In [36]:
selected_reads = data[data["original_label"]==11].copy()

In [37]:
selected_reads.shape

(50826, 30)

In [38]:
selected_reads_prepared = atlas.prepare_reads(selected_reads, atlas=atlas)

In [39]:
input_ceflieish = build_celfieish_input(selected_reads_prepared, atlas)
input_ceflie = build_celfie_input(selected_reads_prepared, atlas)

In [40]:
results = celfieish_deconvolution(input_ceflieish["matrices"], atlas.get_beta_for_regions(input_ceflieish["region_names"]), num_iterations=400, convergence_criteria=0.001)

In [41]:
np.set_printoptions(precision=3)
with np.printoptions(precision=3, suppress=True):
    print(results)

[0.    0.159 0.    0.    0.066 0.    0.364 0.    0.    0.    0.    0.108
 0.    0.17  0.    0.132 0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.   ]


In [42]:
results = celfie_deconvolution(input_ceflie['x_meth'], input_ceflie['x_cov'], *atlas.get_meth_cov_for_regions(input_ceflie["region_names"]), num_iterations=400, convergence_criteria=0.001)

/vsc-hard-mounts/leuven-data/370/vsc37069/usr/methyldl/baselines/deconvolution/celfie/celfie.py:134: RuntimeWarning: invalid value encountered in divide
  _add_pseudocounts(1, np.nan_to_num(y / y_depths), y, y_depths)
/vsc-hard-mounts/leuven-data/370/vsc37069/usr/methyldl/baselines/deconvolution/celfie/celfie.py:135: RuntimeWarning: invalid value encountered in divide
  _add_pseudocounts(0, np.nan_to_num(y / y_depths), y, y_depths)


In [43]:
np.set_printoptions(precision=3)
with np.printoptions(precision=3, suppress=True):
    print(results)

[0.    0.225 0.    0.    0.094 0.019 0.235 0.    0.    0.    0.    0.247
 0.    0.001 0.    0.153 0.    0.    0.    0.    0.    0.    0.    0.026
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.   ]
